We're gonna import the dataset we inspected in notebook 01, but just the first 10k examples, since in these notebooks we're just exploring our possibilites. We're going to use the whole dataset later on.

In [24]:
from datasets import load_dataset
train_dataset = load_dataset(
  "json", 
  data_files="./../data/dataset/python/train.jsonl",
  split="train"
)
train_dataset = train_dataset.select(range(10000))  # Limit to first 10k samples for faster processing

We now have to start tokenizing our text. To do so, for now, we're choosing the [Huggin Face tokenizers](https://github.com/huggingface/tokenizers). We're going to use a BPE-Tokenizer, as suggested by a [paper](https://www.researchgate.net/publication/394590942_How_Different_Tokenization_Algorithms_Impact_LLMs_and_Transformer_Models_for_Binary_Code_Analysis) published in 2025. 

ELIMINA DOPO QUELLO SOPRA O SPOSTA IN ALTRO NOTEBOOK

In our dataset we saw that we actually have a column for both code and summarization (docstring) already tokenized. Now, it is probably not going to be the best option, but for now we're gonna build a vocabolary using the already tokenized strings given by our dataset. We're gonna then compare this approach to using e BPE Tokenizer fully trained. 

In [25]:
code_tokens = train_dataset['code_tokens']

In [26]:
code_tokens[0]

['def',
 'split_phylogeny',
 '(',
 'p',
 ',',
 'level',
 '=',
 '"s"',
 ')',
 ':',
 'level',
 '=',
 'level',
 '+',
 '"__"',
 'result',
 '=',
 'p',
 '.',
 'split',
 '(',
 'level',
 ')',
 'return',
 'result',
 '[',
 '0',
 ']',
 '+',
 'level',
 '+',
 'result',
 '[',
 '1',
 ']',
 '.',
 'split',
 '(',
 '";"',
 ')',
 '[',
 '0',
 ']']

We can now build a vocabolary using gensim's dictionary

In [27]:
from gensim import corpora
code_dictionary = corpora.Dictionary(code_tokens)
print(code_dictionary)

Dictionary<68068 unique tokens: ['";"', '"__"', '"s"', '(', ')']...>


Now that we've created a dictionary let's analyze it a bit.

In [28]:
code_dictionary.num_docs

10000

In [29]:
code_dictionary.num_pos

956632

In [30]:
code_dictionary.token2id

{'";"': 0,
 '"__"': 1,
 '"s"': 2,
 '(': 3,
 ')': 4,
 '+': 5,
 ',': 6,
 '.': 7,
 '0': 8,
 '1': 9,
 ':': 10,
 '=': 11,
 '[': 12,
 ']': 13,
 'def': 14,
 'level': 15,
 'p': 16,
 'result': 17,
 'return': 18,
 'split': 19,
 'split_phylogeny': 20,
 '"""An error occurred trying to create the output directory\n                           ({}) with message: {}"""': 21,
 '"""One or more directories in the path ({}) do not exist. If\n                           you are specifying a new directory for output, please ensure\n                           all other directories in the path currently exist."""': 22,
 '# ENOENT: No such file or directory': 23,
 '# should not happen with os.makedirs': 24,
 '==': 25,
 'ENOENT': 26,
 'OSError': 27,
 'as': 28,
 'd': 29,
 'else': 30,
 'ensure_dir': 31,
 'errno': 32,
 'except': 33,
 'exists': 34,
 'format': 35,
 'if': 36,
 'makedirs': 37,
 'msg': 38,
 'not': 39,
 'oe': 40,
 'os': 41,
 'path': 42,
 'strerror': 43,
 'try': 44,
 'twdd': 45,
 '"Input file is closed."':

In [31]:
vocab = list(code_dictionary.token2id.keys())
vocab[:10]

['";"', '"__"', '"s"', '(', ')', '+', ',', '.', '0', '1']

In [32]:
code_dictionary.cfs

{14: 10578,
 20: 1,
 3: 79370,
 16: 612,
 6: 59823,
 15: 149,
 11: 51567,
 2: 3,
 4: 79370,
 10: 44845,
 5: 4601,
 1: 11,
 17: 1303,
 7: 71569,
 19: 958,
 18: 10497,
 12: 22114,
 8: 4938,
 13: 22114,
 9: 4989,
 0: 9,
 31: 2,
 29: 551,
 36: 15212,
 39: 4208,
 41: 1561,
 42: 2102,
 34: 225,
 44: 1586,
 37: 54,
 33: 1652,
 27: 73,
 28: 1159,
 40: 2,
 24: 1,
 23: 1,
 32: 38,
 25: 3538,
 26: 3,
 38: 644,
 45: 2,
 22: 1,
 35: 2133,
 30: 4415,
 21: 1,
 43: 4,
 53: 3,
 54: 6,
 57: 204,
 47: 2,
 55: 88,
 48: 7433,
 56: 1456,
 52: 543,
 50: 10,
 59: 2484,
 49: 818,
 46: 1,
 51: 1626,
 60: 1633,
 58: 628,
 90: 3,
 95: 16,
 91: 301,
 82: 39,
 67: 1,
 98: 3064,
 111: 2504,
 63: 12,
 72: 5,
 106: 533,
 103: 697,
 112: 2504,
 80: 7,
 97: 698,
 79: 18,
 89: 6615,
 96: 9003,
 76: 1726,
 61: 38,
 109: 156,
 73: 77,
 85: 42,
 87: 94,
 92: 1996,
 88: 313,
 81: 6,
 74: 327,
 66: 1,
 69: 1,
 107: 27,
 105: 362,
 99: 686,
 62: 38,
 100: 1273,
 83: 22,
 108: 7,
 64: 1,
 84: 3,
 101: 1027,
 86: 459,
 94: 2,
 1

We need to add some special tokens, such as UNK and PAD

In [33]:
special_tokens = {'[UNK]': 0, '[PAD]': 1}
code_dictionary.patch_with_special_tokens(special_tokens)
code_dictionary.token2id

{'";"': 68068,
 '"__"': 68069,
 '"s"': 2,
 '(': 3,
 ')': 4,
 '+': 5,
 ',': 6,
 '.': 7,
 '0': 8,
 '1': 9,
 ':': 10,
 '=': 11,
 '[': 12,
 ']': 13,
 'def': 14,
 'level': 15,
 'p': 16,
 'result': 17,
 'return': 18,
 'split': 19,
 'split_phylogeny': 20,
 '"""An error occurred trying to create the output directory\n                           ({}) with message: {}"""': 21,
 '"""One or more directories in the path ({}) do not exist. If\n                           you are specifying a new directory for output, please ensure\n                           all other directories in the path currently exist."""': 22,
 '# ENOENT: No such file or directory': 23,
 '# should not happen with os.makedirs': 24,
 '==': 25,
 'ENOENT': 26,
 'OSError': 27,
 'as': 28,
 'd': 29,
 'else': 30,
 'ensure_dir': 31,
 'errno': 32,
 'except': 33,
 'exists': 34,
 'format': 35,
 'if': 36,
 'makedirs': 37,
 'msg': 38,
 'not': 39,
 'oe': 40,
 'os': 41,
 'path': 42,
 'strerror': 43,
 'try': 44,
 'twdd': 45,
 '"Input file is cl

In [34]:
input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in code_tokens
]
print(input_ids[0])

[14, 20, 3, 16, 6, 15, 11, 2, 4, 10, 15, 11, 15, 5, 68069, 17, 11, 16, 7, 19, 3, 15, 4, 18, 17, 12, 8, 13, 5, 15, 5, 17, 12, 9, 13, 7, 19, 3, 68068, 4, 12, 8, 13]


Now we're quickly going to do the same thing for docstring_tokens, the summarization.

In [36]:
docstring_tokens = train_dataset['docstring_tokens']
docstring_dictionary = corpora.Dictionary(docstring_tokens)
docstring_dictionary.patch_with_special_tokens(special_tokens)
labels = [
  [docstring_dictionary.token2id.get(token, docstring_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in docstring_tokens
]
print(docstring_tokens[0])
print(labels[0])

['Return', 'either', 'the', 'full', 'or', 'truncated', 'version', 'of', 'a', 'QIIME', '-', 'formatted', 'taxonomy', 'string', '.']
[3, 5, 12, 7, 9, 13, 14, 8, 4, 2, 10014, 6, 11, 10, 10015]


We're now going to tokenize test and validation sets too, using the same vocabs we built on the train set.

Let's combine the two sets of data and save them on disk

In [ ]:
from datasets import Dataset, DatasetDict
tokenized_datasets = DatasetDict({
  'train': Dataset.from_dict({
    'input_ids': input_ids,
    'labels': labels
  })
})